In [18]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/churn_cleaned.csv')
print(df.shape)
df.dtypes

(7032, 20)


gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                 int64
dtype: object

In [19]:
categorical_cols = df.select_dtypes(include='str').columns.tolist()
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

print('Categorical:', categorical_cols)
print('Numerical:', numerical_cols)

Categorical: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Numerical: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [20]:
# binary cols get label encoding
binary_cols = [col for col in categorical_cols if df[col].nunique() == 2]

print('Binary Columns:', binary_cols)

for col in binary_cols:
    df[col] = df[col].map({df[col].unique()[0]: 0,
                           df[col].unique()[1]: 1}
                         )
    

Binary Columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']


In [21]:
df.loc[:, binary_cols]

,gender,Partner,Dependents,PhoneService,PaperlessBilling
0,0,0,0,0,0
1,1,1,0,1,1
2,1,1,0,1,0
3,1,1,0,0,1
4,0,1,0,1,0
...,...,...,...,...,...
7027,1,0,1,1,0
7028,0,0,1,1,0
7029,0,0,1,0,0
7030,1,0,0,1,0


In [22]:
# multi-category columns get one-hot encoding
multi_cols = [col for col in categorical_cols 
              if df[col].nunique() > 2]

print("Multi-category columns:", multi_cols)


Multi-category columns: ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']


In [23]:
df = pd.get_dummies(df, columns=multi_cols, drop_first=True)
print('Shape after encoding:', df.shape)

Shape after encoding: (7032, 31)


In [24]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,0,0,1,0,0,29.85,29.85,0,...,False,False,False,False,False,False,False,False,True,False
1,1,0,1,0,34,1,1,56.95,1889.50,0,...,False,False,False,False,False,True,False,False,False,True
2,1,0,1,0,2,1,0,53.85,108.15,1,...,False,False,False,False,False,False,False,False,False,True
3,1,0,1,0,45,0,1,42.30,1840.75,0,...,True,False,False,False,False,True,False,False,False,False
4,0,0,1,0,2,1,0,70.70,151.65,1,...,False,False,False,False,False,False,False,False,True,False


In [25]:
df.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges',
       'Churn', 'MultipleLines_No phone service', 'MultipleLines_Yes',
       'InternetService_Fiber optic', 'InternetService_No',
       'OnlineSecurity_No internet service', 'OnlineSecurity_Yes',
       'OnlineBackup_No internet service', 'OnlineBackup_Yes',
       'DeviceProtection_No internet service', 'DeviceProtection_Yes',
       'TechSupport_No internet service', 'TechSupport_Yes',
       'StreamingTV_No internet service', 'StreamingTV_Yes',
       'StreamingMovies_No internet service', 'StreamingMovies_Yes',
       'Contract_One year', 'Contract_Two year',
       'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check'],
      dtype='str')

In [26]:
# Scale numerical features:

In [27]:
from sklearn.preprocessing import StandardScaler

In [28]:
scale_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

scaler = StandardScaler()

In [29]:
df[scale_cols] = scaler.fit_transform(df[scale_cols])

In [30]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,0,0,-1.280248,0,0,-1.161694,-0.994194,0,...,False,False,False,False,False,False,False,False,True,False
1,1,0,1,0,0.064303,1,1,-0.260878,-0.173740,0,...,False,False,False,False,False,True,False,False,False,True
2,1,0,1,0,-1.239504,1,0,-0.363923,-0.959649,1,...,False,False,False,False,False,False,False,False,False,True
3,1,0,1,0,0.512486,0,1,-0.747850,-0.195248,0,...,True,False,False,False,False,True,False,False,False,False
4,0,0,1,0,-1.239504,1,0,0.196178,-0.940457,1,...,False,False,False,False,False,False,False,False,True,False


In [33]:
# split into features and target:

X = df.drop(columns=['Churn'])
y = df['Churn']

print('Features shape:', X.shape)
print('Target shape:', y.shape)

Features shape: (7032, 30)
Target shape: (7032,)


In [34]:
# Train test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,       # 80% train, 20% test
    random_state=42,     # reproducibility
    stratify=y           # preserve class ratio in both splits
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print("\nChurn rate in train:", y_train.mean().round(3))
print("Churn rate in test:", y_test.mean().round(3))

Train size: (5625, 30)
Test size: (1407, 30)

Churn rate in train: 0.266
Churn rate in test: 0.266


In [35]:
import joblib

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# Save the scaler — we'll need it in production to scale new inputs
joblib.dump(scaler, '../src/models/scaler.pkl')

print("All files saved.")

All files saved.
